# Atmotube Device Report 


This notebook demonstrates how to analyze data from multiple devices of the following type/s: Atmotube. 

## Load Devices

In [1]:
import sys
sys.path.insert(0, "..") # run at repo root
from src.utils import scroll_output # just for output presenation

from src.etl.load import load # calls etl pipeline

MOUNT_PATH = "/home/yul/mnt/proton-data"

# data = load(MOUNT_PATH) # normal usage

data = scroll_output(load, MOUNT_PATH)

In [2]:
from src.utils import consolidate_device_ids, scroll_output

atmotube_consolidated = consolidate_device_ids(data, "Atmotube") # merges all files into one entry per physical device, for Atmotube

list_atmo_ids = list(atmotube_consolidated.keys()) # checking
scroll_output(lambda: print(list_atmo_ids))

## Data Loss

This section summarizes each numeric variable per device within a given time range (manually set to filter).

- `rows`: how many total records were present for that device/variable in the date range.
- `missing`: how many of those records are absent/NA for the variable (after filtering).
- `coverage`: the fraction of available (non-missing) samples:  
  `coverage = (rows - missing) / rows * 100`

### How to read
Higher coverage means the descriptive stats are based on more complete data.
Lower coverage means global/rolling summaries may be less representative (and can change noticeably depending on whether missingness is random or clustered in time).

In [6]:
from src.stats import report_loss, filter_date_range

START = "2026-06-15"
END = "2026-06-30"

def analyze_atmotube():
    print(f"====== {START} to {END} ======")
    print("")

    for true_id, device in atmotube_consolidated.items():
        print(f"=== DEVICE {true_id} ===")
        filtered_device = filter_date_range(device, start=START, end=END)
        report_loss(filtered_device)

scroll_output(analyze_atmotube)

## Descriptive Stats

This section summarizes each numeric variable per device within a given time range (manually set to filter).

1. *Global stats* (global_*)
These describe the variable **across the entire time range**: minimum, 25th percentile, median, mean, 75th percentile, and maximum.

2. *Rolling stats* (roll_*)
These focus on **local behavior over time** by using a time-based rolling window (e.g., `10D`), but within the same time range:

- `roll_min_med`, `roll_center_med`, `roll_max_med` are **typical values of the rolling band** (computed as the median across time of rolling min/center/max).
- `roll_center_mean` is the **average** of the rolling center across time.
- `roll_window_min` / `roll_window_max` are the **extremes ever observed** in the rolling min/max band over the whole period.

### How to read:
- **`n`**: number of non-missing samples used for that variable (after date filtering).
- **`global_*`**: overall distribution for the full period.
- **`roll_*`**: distribution of values computed inside rolling time windows—useful for spotting short-term shifts even when global stats look stable.


In [7]:
from src.stats import report_ranges, filter_date_range
from IPython.display import HTML, display

START = "2026-06-15"
END = "2026-06-30"

def analyze_atmotube():
    display(HTML(f"<b>====== {START} to {END} ======</b>"))
    display(HTML("<br>"))

    for true_id, device in atmotube_consolidated.items():
        display(HTML(f"<b>=== DEVICE {true_id} ===</b>"))
        filtered_device = filter_date_range(device, start=START, end=END)
        df_out = report_ranges(filtered_device, window="10D", center="median")
        display(df_out)
        display(HTML("<br>"))

scroll_output(analyze_atmotube)


,df,column,n,global_min,global_q25,global_med,global_mean,global_q75,global_max,roll_min_med,roll_center_med,roll_max_med,roll_center_mean,roll_window_min,roll_window_max
0,gis,longitude,4565,88.3260,88.36749,88.39685,88.391621,88.40958,88.47491,88.32600,88.395420,88.47491,88.389148,88.3260,88.47491
1,gis,latitude,4565,22.4914,22.54169,22.54630,22.554644,22.56688,22.66101,22.49659,22.546575,22.64826,22.543859,22.4914,22.66101
2,gis,altitude,4565,0.0000,0.00000,0.00000,0.000000,0.00000,0.00000,0.00000,0.000000,0.00000,0.000000,0.0000,0.00000
3,pm,pm1_0_ugm3_atm,10471,0.0000,0.40000,12.10000,16.359374,28.60000,282.60000,0.00000,17.700000,167.60000,17.485503,0.0000,282.60000
4,pm,pm2_5_ugm3_atm,10471,0.0000,0.40000,13.10000,17.477433,30.40000,298.80000,0.00000,18.700000,177.20000,18.617692,0.0000,298.80000
5,pm,pm10_ugm3_atm,10471,0.0000,0.50000,13.30000,17.700583,30.60000,298.80000,0.00000,18.800000,177.20000,18.727777,0.0000,298.80000
6,weather,aqs_total,16036,0.0000,45.00000,60.00000,57.039411,76.00000,100.00000,0.00000,59.000000,92.00000,59.024757,0.0000,100.00000
7,weather,temp_c,14374,24.4000,30.10000,31.50000,31.687352,33.70000,43.60000,24.50000,33.000000,43.60000,32.759037,24.4000,43.60000
8,weather,hum_pct,14374,30.0000,64.00000,69.00000,68.010992,74.00000,86.00000,30.00000,65.000000,86.00000,65.287846,30.0000,86.00000
9,weather,press_hpa,16036,998.5000,1002.10000,1003.50000,1003.337322,1004.50000,1007.70000,1000.40000,1003.700000,1007.70000,1003.627681,998.5000,1007.70000


## Plot Ranges

What does the data for each device look like across time?

In [3]:
from src.stats import filter_date_range, plot_ranges

START = "2026-06-15"
END = "2026-06-30"

# Define the device IDs you want to analyze
device_ids = list(atmotube_consolidated.keys())

In [ ]:
device_index = 0  # Change this to 0, 1, 2, etc. for different devices

true_id = device_ids[device_index]
device = atmotube_consolidated[true_id]

print(f"=== DEVICE {true_id} ({START} to {END}) ===")
filtered_device = filter_date_range(device, start=START, end=END)
plot_ranges(filtered_device)
